In [13]:
import os
import pandas as pd
import numpy as np
import ast

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

from src.utils import get_base_dir


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)

In [4]:
dataset = "dooway"
region = "Basilicata" #"EmiliaRomagna"
region_str =  region.lower() #"emilia_romagna"
base_dir = os.path.join(get_base_dir(), region)
nodes_dir = os.path.join(base_dir, "original_data", "nodes")
edges_dir = os.path.join(base_dir, "original_data", "edges")
feat_dir = os.path.join(base_dir, "heterodata", "features", "dataframes")
edgelist_dir = os.path.join(base_dir, "heterodata", "edgelists")

BASILICATA

In [3]:
def encoding_short_text(df, col, target_dim=28):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    # df[col+'_encoded'] = df.apply(lambda x: model.encode(x[col]), axis=1)

    # Convert the column to string if it's categorical
    if isinstance(df[col].dtype, pd.CategoricalDtype):
        # if pd.api.types.is_categorical_dtype(df[col]):
        df[col] = df[col].astype(str)

    # Generate embeddings for text entries
    def get_embedding(text_or_list):
        ''' if pd.isna(text_or_list) or text_or_list is None:
            return None '''
        if isinstance(text_or_list, str):
            if text_or_list == "":
                return None
            return model.encode(text_or_list)
        if isinstance(text_or_list, list):
            embeddings = [model.encode(text) for text in text_or_list if text != ""]
            if embeddings:
                return np.mean(embeddings, axis=0)
            return None
        return None

    embeddings = df[col].apply(get_embedding)
    # non_nan_embeddings = embeddings.dropna().values
    non_nan_embeddings = [e for e in embeddings if e is not None]

    if len(non_nan_embeddings) > 0:
        embeddings_matrix = np.vstack(non_nan_embeddings)
        mean_embedding = np.mean(embeddings_matrix, axis=0)

        def replace_missing(embedding):
            if embedding is None:
                return mean_embedding
            else:
                return embedding

        embeddings = embeddings.apply(replace_missing)

    embeddings_matrix = np.vstack(embeddings)
    pca = PCA(n_components=target_dim)
    reduced_embeddings = pca.fit_transform(embeddings_matrix)
    df[col + '_encoded'] = [embedding.tolist() for embedding in
                            reduced_embeddings]  # Store the reduced embeddings in a single column as lists
    df.drop([col], axis=1, inplace=True)
    return df

In [4]:
def compute_italian_embedding(df, text_column):
    
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    mask_valid = df[text_column].notna()
    valid_indices = df.loc[mask_valid].index
    valid_texts = df.loc[valid_indices, text_column].tolist()
    embeddings = model.encode(valid_texts, show_progress_bar=True, convert_to_numpy=True)
    new_column_name = text_column + '_embedding'
    df[new_column_name] = None

    # Assegna ogni embedding riga per riga in modo sicuro
    for idx, emb in zip(valid_indices, embeddings):
        df.at[idx, new_column_name] = emb.tolist()

    return df

In [5]:
def count_nan_or_empty(series):
    return series.apply(lambda x: (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and (x == '[]' or x == 'none'))).sum()


def drop_columns_with_many_nans(df):
    threshold = len(df) / 2
    columns_to_drop = [col for col in df.columns if count_nan_or_empty(df[col]) > threshold]
    df_cleaned = df.drop(columns=columns_to_drop)
    return df_cleaned

In [6]:
def one_hot_encoding(df, col, exclude_values=None):
    unique_values = df[col].unique()
    if exclude_values:
        unique_values = [val for val in unique_values if val not in exclude_values]
    for val in unique_values:
        col_name = val.rstrip('s') if val.endswith('s') else val
        df[col_name] = df[col] == val
    df.drop([col], axis=1, inplace=True)
    return df
    

In [7]:
def safe_parse_json(val):
        if pd.isna(val) or not isinstance(val, str):
            return {}
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return {}

CATEGORY

In [8]:
file_path = os.path.join(nodes_dir, "category.parquet")

with open(file_path, "rb") as f:
    head = f.read(4)
    f.seek(-4, 2)
    tail = f.read(4)

print(f"Start bytes: {head}, End bytes: {tail}")

Start bytes: b'PAR1', End bytes: b'PAR1'


In [9]:
df = pd.read_parquet(os.path.join(nodes_dir, "category.parquet"))
df.head()

,unique_id,category
0,0,cycling_park
1,1,solar_photovoltaic_power_plant
2,2,art_center
3,3,electronics_company
4,4,natural_history_museum


In [6]:
df.shape

(1525, 2)

In [10]:
df["category"] = df["category"].apply(lambda x: x.replace("_", " "))
df.head()

,unique_id,category
0,0,cycling park
1,1,solar photovoltaic power plant
2,2,art center
3,3,electronics company
4,4,natural history museum


In [11]:
df = encoding_short_text(df, "category", target_dim=256)
df.head()
df.to_parquet(os.path.join(feat_dir, "category.parquet"), index=False)

In [10]:
test = pd.read_parquet(os.path.join(feat_dir, "category.parquet"))
test.head()


,unique_id,category_encoded
0,0,"[-0.3043428361415863, -0.29231059551239014, 0...."
1,1,"[-0.10251842439174652, 0.13989387452602386, 0...."
2,2,"[-0.1398950219154358, -0.16000044345855713, 0...."
3,3,"[0.15114553272724152, 0.2636007070541382, -0.0..."
4,4,"[-0.11031119525432587, -0.12446992844343185, 0..."


POI

In [12]:
df = pd.read_parquet(os.path.join(nodes_dir, "poi.parquet"))
# df.shape (14853, 35) --- to drop: REVIEWS, SENTIMENT, INDUSTRY, CUSTOM_CATEGORY
df.head()


,index,type,rank_group,rank_absolute,domain,title,url,contact_url,rating,hotel_rating,...,local_justifications,is_directory_item,contributor_url,region,reviews,sentiment,industry,custom_category,unique_id,main_image_embedding
0,0,restaurants,1,1,None,L'Antico Borgo,None,None,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",NaN,...,None,False,None,basilicata,NaN,NaN,NaN,NaN,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560..."
1,1,restaurants,2,2,www.chin-up.it,"Chin Up! ""Al nido del falco""",http://www.chin-up.it/,None,"{'rating_type': 'Max5', 'value': 4.7, 'votes_c...",NaN,...,None,False,https://maps.google.com/maps/contrib/100396939...,basilicata,NaN,NaN,NaN,NaN,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835..."
2,2,restaurants,3,3,None,Antico borgo di Stolfi Angelo,None,None,"{'rating_type': 'Max5', 'value': 4.5, 'votes_c...",NaN,...,None,False,https://maps.google.com/maps/contrib/104704191...,basilicata,NaN,NaN,NaN,NaN,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752..."
3,3,restaurants,4,4,None,Central Bar Pub,None,None,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",NaN,...,None,False,https://maps.google.com/maps/contrib/108493660...,basilicata,NaN,NaN,NaN,NaN,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24..."
4,4,restaurants,5,5,www.hotelsellata.it,Hotel Sellata,http://www.hotelsellata.it/,None,"{'rating_type': 'Max5', 'value': 3.8, 'votes_c...",3.0,...,"[{'type': 'user_review', 'text': '""Ottimo post...",False,None,basilicata,NaN,NaN,NaN,NaN,4,None


In [64]:
df.shape

(14853, 36)

In [13]:
df_cleaned = drop_columns_with_many_nans(df)
df_cleaned.shape

(14853, 31)

In [85]:
df_cleaned.columns

Index(['index', 'type', 'rank_group', 'rank_absolute', 'domain', 'title',
       'url', 'contact_url', 'rating', 'price_level', 'rating_distribution',
       'address', 'address_info', 'place_id', 'main_image', 'total_photos',
       'category', 'additional_categories', 'category_ids', 'work_hours',
       'feature_id', 'cid', 'latitude', 'longitude', 'is_claimed',
       'local_justifications', 'is_directory_item', 'contributor_url',
       'region', 'unique_id', 'main_image_embedding'],
      dtype='object')

In [14]:
other_columns_to_drop = ["index", "region", "main_image", "rank_group", "place_id", "feature_id", "cid", 
                         "domain", "url", "contact_url", "contributor_url", "is_directory_item",
                         "rating_distribution", "address", "address_info", "local_justifications",
                         "category", "additional_categories", "category_ids"]
df_cleaned = df_cleaned.drop(columns=other_columns_to_drop)

In [15]:
df_cleaned.columns

Index(['type', 'rank_absolute', 'title', 'rating', 'price_level',
       'total_photos', 'work_hours', 'latitude', 'longitude', 'is_claimed',
       'unique_id', 'main_image_embedding'],
      dtype='object')

In [16]:
df_cleaned.head()

,type,rank_absolute,title,rating,price_level,total_photos,work_hours,latitude,longitude,is_claimed,unique_id,main_image_embedding
0,restaurants,1,L'Antico Borgo,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",None,1.0,"{'timetable': None, 'current_status': 'close'}",40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560..."
1,restaurants,2,"Chin Up! ""Al nido del falco""","{'rating_type': 'Max5', 'value': 4.7, 'votes_c...",None,427.0,"{'timetable': {'sunday': [{'open': {'hour': 9,...",40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835..."
2,restaurants,3,Antico borgo di Stolfi Angelo,"{'rating_type': 'Max5', 'value': 4.5, 'votes_c...",None,4.0,"{'timetable': None, 'current_status': 'close'}",40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752..."
3,restaurants,4,Central Bar Pub,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",inexpensive,4.0,"{'timetable': {'sunday': [{'open': {'hour': 0,...",40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24..."
4,restaurants,5,Hotel Sellata,"{'rating_type': 'Max5', 'value': 3.8, 'votes_c...",None,93.0,"{'timetable': None, 'current_status': 'close'}",40.5109644,15.8063664,False,4,None


In [17]:
df_cleaned = one_hot_encoding(df_cleaned, "type", exclude_values=["unknown"])
df_cleaned.head()

,rank_absolute,title,rating,price_level,total_photos,work_hours,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction
0,1,L'Antico Borgo,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",None,1.0,"{'timetable': None, 'current_status': 'close'}",40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False
1,2,"Chin Up! ""Al nido del falco""","{'rating_type': 'Max5', 'value': 4.7, 'votes_c...",None,427.0,"{'timetable': {'sunday': [{'open': {'hour': 9,...",40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False
2,3,Antico borgo di Stolfi Angelo,"{'rating_type': 'Max5', 'value': 4.5, 'votes_c...",None,4.0,"{'timetable': None, 'current_status': 'close'}",40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False
3,4,Central Bar Pub,"{'rating_type': 'Max5', 'value': 4.6, 'votes_c...",inexpensive,4.0,"{'timetable': {'sunday': [{'open': {'hour': 0,...",40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False
4,5,Hotel Sellata,"{'rating_type': 'Max5', 'value': 3.8, 'votes_c...",None,93.0,"{'timetable': None, 'current_status': 'close'}",40.5109644,15.8063664,False,4,None,True,False,False


In [18]:
df_cleaned["rating"] = df_cleaned["rating"].apply(safe_parse_json)
df_cleaned["avg_rating"] = df_cleaned["rating"].apply(lambda x: float(x.get('value', 0.0)) if x.get('value') is not None else 0.0)
df_cleaned['votes_count'] = df_cleaned["rating"].apply(lambda x: float(x.get('votes_count', 0.0)) if x.get('votes_count') is not None else 0.0)
df_cleaned.drop(["rating"], axis=1, inplace=True)
df_cleaned.head()


,rank_absolute,title,price_level,total_photos,work_hours,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count
0,1,L'Antico Borgo,None,1.0,"{'timetable': None, 'current_status': 'close'}",40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31.0
1,2,"Chin Up! ""Al nido del falco""",None,427.0,"{'timetable': {'sunday': [{'open': {'hour': 9,...",40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217.0
2,3,Antico borgo di Stolfi Angelo,None,4.0,"{'timetable': None, 'current_status': 'close'}",40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2.0
3,4,Central Bar Pub,inexpensive,4.0,"{'timetable': {'sunday': [{'open': {'hour': 0,...",40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41.0
4,5,Hotel Sellata,None,93.0,"{'timetable': None, 'current_status': 'close'}",40.5109644,15.8063664,False,4,None,True,False,False,3.8,61.0


In [19]:
df_cleaned = one_hot_encoding(df_cleaned, "price_level", exclude_values=[None])
df_cleaned.head()

,rank_absolute,title,total_photos,work_hours,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive
0,1,L'Antico Borgo,1.0,"{'timetable': None, 'current_status': 'close'}",40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31.0,False,False,False,False
1,2,"Chin Up! ""Al nido del falco""",427.0,"{'timetable': {'sunday': [{'open': {'hour': 9,...",40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217.0,False,False,False,False
2,3,Antico borgo di Stolfi Angelo,4.0,"{'timetable': None, 'current_status': 'close'}",40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2.0,False,False,False,False
3,4,Central Bar Pub,4.0,"{'timetable': {'sunday': [{'open': {'hour': 0,...",40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41.0,True,False,False,False
4,5,Hotel Sellata,93.0,"{'timetable': None, 'current_status': 'close'}",40.5109644,15.8063664,False,4,None,True,False,False,3.8,61.0,False,False,False,False


In [20]:
def check_timetable(val):
    if pd.isna(val):
        return True
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return True
    if isinstance(val, dict):
        return val.get('timetable') is None
    return True 

df_cleaned["open24/7"] = df_cleaned["work_hours"].apply(check_timetable)
df_cleaned.drop(["work_hours"], axis=1, inplace=True)
df_cleaned.head()

,rank_absolute,title,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive,open24/7
0,1,L'Antico Borgo,1.0,40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31.0,False,False,False,False,True
1,2,"Chin Up! ""Al nido del falco""",427.0,40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217.0,False,False,False,False,False
2,3,Antico borgo di Stolfi Angelo,4.0,40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2.0,False,False,False,False,True
3,4,Central Bar Pub,4.0,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41.0,True,False,False,False,False
4,5,Hotel Sellata,93.0,40.5109644,15.8063664,False,4,None,True,False,False,3.8,61.0,False,False,False,False,True


In [21]:
df_cleaned = compute_italian_embedding(df_cleaned, "title")
df_cleaned.drop(["title"], axis=1, inplace=True)
df_cleaned.head()

Batches: 100%|██████████| 465/465 [00:23<00:00, 19.73it/s]


,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive,open24/7,title_embedding
0,1,1.0,40.5080386,15.8128442,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31.0,False,False,False,False,True,"[-0.0924561396241188, 0.3570820093154907, -0.1..."
1,2,427.0,40.5228839,15.7858469,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217.0,False,False,False,False,False,"[0.03634057939052582, 0.37386876344680786, 0.0..."
2,3,4.0,40.5087182,15.8112784,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2.0,False,False,False,False,True,"[0.2866106331348419, 0.33424651622772217, -0.0..."
3,4,4.0,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41.0,True,False,False,False,False,"[0.6681708693504333, -0.4947139322757721, -0.1..."
4,5,93.0,40.5109644,15.8063664,False,4,None,True,False,False,3.8,61.0,False,False,False,False,True,"[0.4959424138069153, -0.09553607553243637, -0...."


In [22]:
def fix_longitude(value):
    if isinstance(value, str):
        parts = value.split('.')
        if len(parts) > 1:
            # Join all parts after the first dot
            fixed = parts[0] + '.' + ''.join(parts[1:])
            try:
                return float(fixed)
            except ValueError:
                return None
    return None  # or return float(value) if already valid

# Apply to longitude column
df_cleaned['longitude'] = df_cleaned['longitude'].apply(fix_longitude)
df_cleaned.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive,open24/7,title_embedding
0,1,1.0,40.5080386,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31.0,False,False,False,False,True,"[-0.0924561396241188, 0.3570820093154907, -0.1..."
1,2,427.0,40.5228839,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217.0,False,False,False,False,False,"[0.03634057939052582, 0.37386876344680786, 0.0..."
2,3,4.0,40.5087182,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2.0,False,False,False,False,True,"[0.2866106331348419, 0.33424651622772217, -0.0..."
3,4,4.0,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41.0,True,False,False,False,False,"[0.6681708693504333, -0.4947139322757721, -0.1..."
4,5,93.0,40.5109644,15.806366,False,4,None,True,False,False,3.8,61.0,False,False,False,False,True,"[0.4959424138069153, -0.09553607553243637, -0...."


In [23]:
df_cleaned["total_photos"] = df_cleaned["total_photos"].fillna(0)

df_cleaned = df_cleaned.astype({"total_photos": int, "latitude": float, "longitude": float, "is_claimed":bool, "votes_count": int})

In [100]:
df_cleaned.dtypes

rank_absolute             int64
total_photos              int64
latitude                float64
longitude               float64
is_claimed                 bool
unique_id                 int64
main_image_embedding     object
restaurant                 bool
hotel                      bool
attraction                 bool
avg_rating              float64
votes_count               int64
inexpensive                bool
moderate                   bool
expensive                  bool
very_expensive             bool
open24/7                   bool
title_embedding          object
dtype: object

In [24]:
df_cleaned.to_parquet(os.path.join(feat_dir, "poi.parquet"), index=False)

In [102]:
test = pd.read_parquet(os.path.join(feat_dir, "poi.parquet"))
test.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive,open24/7,title_embedding
0,1,1,40.508039,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31,False,False,False,False,True,"[-0.0924561396241188, 0.3570820093154907, -0.1..."
1,2,427,40.522884,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217,False,False,False,False,False,"[0.03634057939052582, 0.37386876344680786, 0.0..."
2,3,4,40.508718,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2,False,False,False,False,True,"[0.2866106331348419, 0.33424651622772217, -0.0..."
3,4,4,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41,True,False,False,False,False,"[0.6681708693504333, -0.4947139322757721, -0.1..."
4,5,93,40.510964,15.806366,False,4,None,True,False,False,3.8,61,False,False,False,False,True,"[0.4959424138069153, -0.09553607553243637, -0...."


In [14]:
# SCORPORARE IMMAGINI DA POIs

df_poi = pd.read_parquet(os.path.join(feat_dir, "poi.parquet"))

df_img = df_poi[["unique_id", "main_image_embedding"]].copy()
df_img_ok = df_img.dropna(subset=["main_image_embedding"])
df_img_ok.to_parquet(os.path.join(feat_dir, "image.parquet"), index=False)

df_img_ok.head()


,unique_id,main_image_embedding
0,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560..."
1,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835..."
2,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752..."
3,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24..."
4,4,"[-0.109207876, 0.08907696, -0.011386456, -0.04..."


In [16]:
df = df_img_ok[["unique_id"]].copy()
df.rename(columns={"unique_id": "unique_id_poi"}, inplace=True)
df["unique_id_image"] = df.index
df.to_parquet(os.path.join(edgelist_dir, "poi_has_main_image_image.parquet"), index=False)
print(df.head())

df_rev = df[["unique_id_image", "unique_id_poi"]].copy()
df_rev.to_parquet(os.path.join(edgelist_dir, "image_is_main_image_of_poi.parquet"), index=False)
print(df_rev.head())

   unique_id_poi  unique_id_image
0              0                0
1              1                1
2              2                2
3              3                3
4              4                4
   unique_id_image  unique_id_poi
0                0              0
1                1              1
2                2              2
3                3              3
4                4              4


REVIEW

In [5]:
df = pd.read_parquet(os.path.join(nodes_dir, "review.parquet"))
# df.shape (1700367, 14)
df.head()

,cid,place_id,location_code,type,datetime,profile_url,review_text,timestamp,rating_value,reviews_count,photos_count,local_guide,review_highlights,unique_id,review_text_embedding
0,6526278041137503800,ChIJdeNxK7UyPxMRON5u-0z_kVo,9053426,google_reviews,2023-12-04 10:39:53 +00:00,QA1VvS65a3xuYX49qQU2BS1hwqjKID2+kqPadWT8oL8=,Paese di montagna con boschi di faggi e castag...,2020-12-04 10:39:16 +00:00,4.0,100.0,274.0,True,None,0,"[-0.02132909931242466, 0.0855952724814415, 0.0..."
1,6526278041137503800,ChIJdeNxK7UyPxMRON5u-0z_kVo,9053426,google_reviews,2023-12-04 10:39:53 +00:00,LTZBZHp0ee1g8sWcfwl9CZRctKfoLkNRbtGlbZCDe08=,"Paese caratteristico con tanto da visitare,il ...",2021-12-04 10:39:16 +00:00,5.0,11.0,115.0,True,None,1,"[0.11749532073736191, 0.027359535917639732, 0...."
2,6526278041137503800,ChIJdeNxK7UyPxMRON5u-0z_kVo,9053426,google_reviews,2023-12-04 10:39:53 +00:00,Tn9Bq3/3GlLrDgcsZ7AgQk9X9oYPR6pSBzYZpiIjm7E=,Concerto Gospel del 23/12/2021\nOrganizzato da...,2022-12-04 10:39:16 +00:00,5.0,18.0,33.0,True,None,2,"[-0.0026724785566329956, 0.1681678593158722, 0..."
3,6526278041137503800,ChIJdeNxK7UyPxMRON5u-0z_kVo,9053426,google_reviews,2023-12-04 10:39:53 +00:00,5gY0Vv43v6oFdy5Op/vqm1V1cPvcU15qIUkruSt1jbw=,Piccolo borgo appollaiato tra i monti lucani.\...,2021-12-04 10:39:16 +00:00,5.0,138.0,21.0,True,None,3,"[0.11633721739053726, -0.0018988546216860414, ..."
4,6526278041137503800,ChIJdeNxK7UyPxMRON5u-0z_kVo,9053426,google_reviews,2023-12-04 10:39:53 +00:00,LG9mZbib8Vqjybu4atSTctvyXtU9rDiKSO3eX0IaX1w=,"Bel paese, ora con il ponte tibetano potrebbe ...",2021-12-04 10:39:16 +00:00,4.0,60.0,17.0,False,None,4,"[-0.1472267359495163, -0.14695429801940918, 0...."


In [6]:
df.shape

(1700367, 15)

In [27]:
df.columns

Index(['cid', 'place_id', 'location_code', 'type', 'datetime', 'profile_url',
       'review_text', 'timestamp', 'rating_value', 'reviews_count',
       'photos_count', 'local_guide', 'review_highlights', 'unique_id',
       'review_text_embedding'],
      dtype='object')

In [7]:
columns_to_drop = ["cid", "place_id", "location_code", "type", "profile_url", "review_text"]
df_cleaned = df.drop(columns=columns_to_drop)
df_cleaned.head()

,datetime,timestamp,rating_value,reviews_count,photos_count,local_guide,review_highlights,unique_id,review_text_embedding
0,2023-12-04 10:39:53 +00:00,2020-12-04 10:39:16 +00:00,4.0,100.0,274.0,True,None,0,"[-0.02132909931242466, 0.0855952724814415, 0.0..."
1,2023-12-04 10:39:53 +00:00,2021-12-04 10:39:16 +00:00,5.0,11.0,115.0,True,None,1,"[0.11749532073736191, 0.027359535917639732, 0...."
2,2023-12-04 10:39:53 +00:00,2022-12-04 10:39:16 +00:00,5.0,18.0,33.0,True,None,2,"[-0.0026724785566329956, 0.1681678593158722, 0..."
3,2023-12-04 10:39:53 +00:00,2021-12-04 10:39:16 +00:00,5.0,138.0,21.0,True,None,3,"[0.11633721739053726, -0.0018988546216860414, ..."
4,2023-12-04 10:39:53 +00:00,2021-12-04 10:39:16 +00:00,4.0,60.0,17.0,False,None,4,"[-0.1472267359495163, -0.14695429801940918, 0...."


In [8]:
df_cleaned = df_cleaned.drop(columns=["datetime"])
df_cleaned["timestamp"] = pd.to_datetime(df_cleaned["timestamp"])

In [9]:
df_cleaned.head()

,timestamp,rating_value,reviews_count,photos_count,local_guide,review_highlights,unique_id,review_text_embedding
0,2020-12-04 10:39:16+00:00,4.0,100.0,274.0,True,None,0,"[-0.02132909931242466, 0.0855952724814415, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,11.0,115.0,True,None,1,"[0.11749532073736191, 0.027359535917639732, 0...."
2,2022-12-04 10:39:16+00:00,5.0,18.0,33.0,True,None,2,"[-0.0026724785566329956, 0.1681678593158722, 0..."
3,2021-12-04 10:39:16+00:00,5.0,138.0,21.0,True,None,3,"[0.11633721739053726, -0.0018988546216860414, ..."
4,2021-12-04 10:39:16+00:00,4.0,60.0,17.0,False,None,4,"[-0.1472267359495163, -0.14695429801940918, 0...."


In [10]:
extracted_rows = []

# Iterate over non-null review_highlights
for _, row in df_cleaned[~df_cleaned["review_highlights"].isna()].iterrows():
    review_id = row["unique_id"]
    highlights = row["review_highlights"]
    
    for highlight in highlights:
        score_str = highlight["assessment"]
        """
        try:
            score = int(score_str.split('/')[0])
        except (ValueError, AttributeError, IndexError):
            score = -1  # fallback if format is unexpected
        """
        extracted_rows.append({
            "review_id": review_id,
            "service": highlight["feature"],
            "score": score_str #score
        })

# Convert to DataFrame
df_extracted = pd.DataFrame(extracted_rows)

In [11]:
df_cleaned = df_cleaned.drop(columns=["reviews_count", "photos_count"])
df_cleaned.head()

,timestamp,rating_value,local_guide,review_highlights,unique_id,review_text_embedding
0,2020-12-04 10:39:16+00:00,4.0,True,None,0,"[-0.02132909931242466, 0.0855952724814415, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,True,None,1,"[0.11749532073736191, 0.027359535917639732, 0...."
2,2022-12-04 10:39:16+00:00,5.0,True,None,2,"[-0.0026724785566329956, 0.1681678593158722, 0..."
3,2021-12-04 10:39:16+00:00,5.0,True,None,3,"[0.11633721739053726, -0.0018988546216860414, ..."
4,2021-12-04 10:39:16+00:00,4.0,False,None,4,"[-0.1472267359495163, -0.14695429801940918, 0...."


In [12]:
df_cleaned.dtypes

timestamp                datetime64[ns, UTC]
rating_value                         float64
local_guide                             bool
review_highlights                     object
unique_id                              int64
review_text_embedding                 object
dtype: object

In [13]:
df_cleaned.to_parquet(os.path.join(feat_dir, "review.parquet"), index=False)

In [58]:
df_extracted.shape

(674947, 3)

In [59]:
df_extracted.head()

,review_id,service,score
0,653221,Cibo,5/5
1,653221,Servizio,5/5
2,653221,Ambiente,5/5
3,653224,Cibo,5/5
4,653224,Servizio,5/5


In [14]:
df_extracted["service"].unique() #24

array(['Cibo', 'Servizio', 'Ambiente', 'Area parcheggio',
       'Opzioni per il parcheggio', 'Positiva', 'Piatti consigliati',
       'Prenotazione', 'Livello di rumore', 'Dimensione gruppo',
       'Consigliato per i vegetariani', 'Piatti vegetariani',
       'Regimi alimentari previsti', 'Tempo di attesa',
       'Tipo di posto a sedere', 'Camere', 'Posizione',
       "Punti forti dell'hotel", 'Eventi speciali', 'Visitato in data',
       'Prenotazione consigliata', 'Negativa', 'Attività consigliate',
       'Servizi'], dtype=object)

In [9]:
to_remove = ["Visitato in data", "Eventi speciali"]

to_service_node_review = ["Cibo", "Servizio", "Ambiente", "Camere", "Posizione", "Consigliato per i vegetariani"] #review_id, service_name, score           
#food, service, atmosphere, rooms, location

to_service_node_poi = ["Area parcheggio", "Opzioni per il parcheggio", "Attività consigliate", "Servizi", "Piatti vegetariani", "Regimi alimentari previsti", "Prenotazione", "Prenotazione consigliata", "Prenotazione"] #review_id, service_name, score    
#parcheggio_gratuito, parcheggio_a_pagamento, passeggiate, escursioni, tour_della_citta, centro_estetico, cucina_vegetariana, cucina_vegana, cucina_gluten_free, cucina_senza_lattosio, prenotazione
#free_parking, paid_parking, walks, excursions, city_tour, beauty_center, vegetarian_cuisine, vegan_cuisine, gluten-free_cuisine, lactose-free_cuisine, reservation

to_description_column = ["Punti forti dell'hotel", "Tipo di posto a sedere", "Piatti consigliati"] #description

to_boolean_column = ["Positiva", "Negativa"] #qualita_positiva, qualita_negativa
#positive_quality, negative_quality

to_categorical_column = ["Livello di rumore", "Dimensione gruppo", "Tempo di attesa"]
#noise_level, group_size, waiting_time




In [21]:
extracted_rows_review = []
extracted_rows_poi = []
extracted_rows_attributes = []


# Iterate over non-null review_highlights
for _, row in df_cleaned[~df_cleaned["review_highlights"].isna()].iterrows():
    review_id = row["unique_id"]
    highlights = row["review_highlights"]
    
    for highlight in highlights:
        service = highlight["feature"]
        score = highlight["assessment"]
        
        new_row = {
            "review_id": review_id,
            "service": service,
            "score": score
        }
        
        if service in to_service_node_review: 
            extracted_rows_review.append(new_row)
        elif service in to_service_node_poi:
            extracted_rows_poi.append(new_row)
        else:
            extracted_rows_attributes.append(new_row)

# Convert to DataFrame
df_extracted_review = pd.DataFrame(extracted_rows_review)
df_extracted_poi = pd.DataFrame(extracted_rows_poi)
df_extracted_attributes = pd.DataFrame(extracted_rows_attributes)

df_extracted_review.to_parquet(os.path.join(feat_dir, "tmp", "review_service.parquet"), index=False)
df_extracted_poi.to_parquet(os.path.join(feat_dir, "tmp", "poi_service.parquet"), index=False)
df_extracted_attributes.to_parquet(os.path.join(feat_dir, "tmp", "review_attrs.parquet"), index=False)


In [13]:
# WORK ON ATTRIBUTES
df_extracted_attributes = pd.read_parquet(os.path.join(feat_dir, "tmp", "review_attrs.parquet"))
df_attrs = df_extracted_attributes[~df_extracted_attributes.isin(to_remove)]
df_attrs = df_attrs.dropna(subset=["service"])

In [14]:
df_attrs["service"].unique()

array(['Positiva', 'Piatti consigliati', 'Livello di rumore',
       'Dimensione gruppo', 'Tempo di attesa', 'Tipo di posto a sedere',
       "Punti forti dell'hotel", 'Negativa'], dtype=object)

In [15]:
df_attrs['positive_quality'] = False
df_attrs['negative_quality'] = False
df_attrs['description'] = ""

df_attrs.loc[df_attrs['service'] == 'Positiva', 'positive_quality'] = True
df_attrs.loc[df_attrs['service'] == 'Negativa', 'negative_quality'] = True
df_attrs.loc[df_attrs['service'].isin(to_description_column), 'description'] = df_attrs['score']

df_attrs.head(20)

,review_id,service,score,positive_quality,negative_quality,description
0,653411,Positiva,Qualità e Professionalità,True,False,
1,653429,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...
2,653430,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...
3,653432,Piatti consigliati,Pokè,False,False,Pokè
4,653436,Piatti consigliati,Pokè,False,False,Pokè
5,653437,Piatti consigliati,Pokè e Panino Kebab,False,False,Pokè e Panino Kebab
6,653446,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...
7,653447,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...
8,653470,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...
9,653478,Piatti consigliati,Tette E Cornetto Farcito,False,False,Tette E Cornetto Farcito


In [17]:
df_attrs[df_attrs["service"] == "Livello di rumore"]["score"].unique()

array(['Alto, ma puoi comunque parlare', 'Molto basso',
       'Basso, facile parlare', 'Rumore moderato', 'Non so'], dtype=object)

In [20]:
to_categorical_column = ["Livello di rumore", "Dimensione gruppo", "Tempo di attesa"]
#noise_level, group_size, waiting_time

noise_order = [
    "Molto basso",
    "Basso, facile parlare",
    "Rumore moderato",
    "Alto, ma puoi comunque parlare"
]
noise_map = {label: i for i, label in enumerate(noise_order)}

def map_noise_level(row):
    if row['service'] == "Livello di rumore" and row['score'] in noise_map:
        return noise_map[row['score']]
    else:
        return None

df_attrs['noise_level'] = df_attrs.apply(map_noise_level, axis=1)
df_attrs['noise_level'] = df_attrs['noise_level'].fillna(-1).astype(int)

df_attrs.head()



,review_id,service,score,positive_quality,negative_quality,description,noise_level
0,653411,Positiva,Qualità e Professionalità,True,False,,-1
1,653429,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1
2,653430,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1
3,653432,Piatti consigliati,Pokè,False,False,Pokè,-1
4,653436,Piatti consigliati,Pokè,False,False,Pokè,-1


In [21]:
df_attrs[df_attrs["service"] == "Dimensione gruppo"]["score"].unique()

array(['Non so', '3-4 persone e Adatto a gruppi di tutte le dimensioni',
       'Adatto a gruppi di tutte le dimensioni', '5-8 persone',
       '1 persona, 2 persone, 3-4 persone e 5-8 persone', '3-4 persone',
       'Più di 9 persone',
       '1 persona, 2 persone, 3-4 persone, 5-8 persone e Più di 9 persone',
       '5-8 persone e Più di 9 persone', '2 persone',
       '1 persona, 2 persone, 3-4 persone, 5-8 persone, Più di 9 persone e Adatto a gruppi di tutte le dimensioni',
       '2 persone e 3-4 persone', '3-4 persone e 5-8 persone',
       '2 persone, 3-4 persone e 5-8 persone'], dtype=object)

In [22]:
def map_group_size(row):
    if row['service'] != "Dimensione gruppo":
        return -1  # not relevant
    
    score = str(row['score'])  # ensure it's a string for matching
    
    if "tutte le dimensioni" in score or "Più di 9" in score:
        return 2
    elif "5-8" in score:
        return 1
    elif "3-4" in score:
        return 0
    else:
        return -1

df_attrs['group_size'] = df_attrs.apply(map_group_size, axis=1)

df_attrs.head()

,review_id,service,score,positive_quality,negative_quality,description,noise_level,group_size
0,653411,Positiva,Qualità e Professionalità,True,False,,-1,-1
1,653429,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1,-1
2,653430,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1,-1
3,653432,Piatti consigliati,Pokè,False,False,Pokè,-1,-1
4,653436,Piatti consigliati,Pokè,False,False,Pokè,-1,-1


In [24]:
df_attrs[df_attrs["service"] == "Tempo di attesa"]["score"].unique()

array(['Nessuna attesa', 'Fino a 10 min', '1 h o più', '30-60 min',
       '10-30 min'], dtype=object)

In [25]:
time_order = [
    "Nessuna attesa",
    "Fino a 10 min",
    "10-30 min",
    "30-60 min",
    "1h o più"
]
time_map = {label: i for i, label in enumerate(time_order)}

def map_waiting_time(row):
    if row['service'] == "Tempo di attesa" and row['score'] in time_map:
        return time_map[row['score']]
    else:
        return None

df_attrs['waiting_time'] = df_attrs.apply(map_waiting_time, axis=1)
df_attrs['waiting_time'] = df_attrs['waiting_time'].fillna(-1).astype(int)

df_attrs.head()

,review_id,service,score,positive_quality,negative_quality,description,noise_level,group_size,waiting_time
0,653411,Positiva,Qualità e Professionalità,True,False,,-1,-1,-1
1,653429,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1,-1,-1
2,653430,Piatti consigliati,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,False,False,Kebab Roll Kebab Box Pollo Crepes Burger Insal...,-1,-1,-1
3,653432,Piatti consigliati,Pokè,False,False,Pokè,-1,-1,-1
4,653436,Piatti consigliati,Pokè,False,False,Pokè,-1,-1,-1


In [26]:
df = compute_italian_embedding(df_attrs, "description")
df.drop(["description","service", "score"], axis=1, inplace=True)
df.head()

Batches: 100%|██████████| 1897/1897 [01:31<00:00, 20.71it/s]


,review_id,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
0,653411,True,False,-1,-1,-1,"[0.22691738605499268, 0.08178411424160004, 0.0..."
1,653429,False,False,-1,-1,-1,"[-0.2566263675689697, 0.1666107475757599, -0.0..."
2,653430,False,False,-1,-1,-1,"[-0.2566264271736145, 0.16661083698272705, -0...."
3,653432,False,False,-1,-1,-1,"[0.0987292230129242, 0.07315211743116379, 0.00..."
4,653436,False,False,-1,-1,-1,"[0.0987292230129242, 0.07315211743116379, 0.00..."


In [27]:
df.to_parquet(os.path.join(feat_dir, "tmp", "review_attrs_processed.parquet"), index=False)

In [28]:
df_big = pd.read_parquet(os.path.join(feat_dir, "review.parquet"))
df_big.head()

,timestamp,rating_value,local_guide,review_highlights,unique_id,review_text_embedding
0,2020-12-04 10:39:16+00:00,4.0,True,None,0,"[-0.02132909931242466, 0.0855952724814415, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,True,None,1,"[0.11749532073736191, 0.027359535917639732, 0...."
2,2022-12-04 10:39:16+00:00,5.0,True,None,2,"[-0.0026724785566329956, 0.1681678593158722, 0..."
3,2021-12-04 10:39:16+00:00,5.0,True,None,3,"[0.11633721739053726, -0.0018988546216860414, ..."
4,2021-12-04 10:39:16+00:00,4.0,False,None,4,"[-0.1472267359495163, -0.14695429801940918, 0...."


In [29]:
df_small = pd.read_parquet(os.path.join(feat_dir, "tmp", "review_attrs_processed.parquet"))
df_small.head()

,review_id,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
0,653411,True,False,-1,-1,-1,"[0.22691738605499268, 0.08178411424160004, 0.0..."
1,653429,False,False,-1,-1,-1,"[-0.2566263675689697, 0.1666107475757599, -0.0..."
2,653430,False,False,-1,-1,-1,"[-0.2566264271736145, 0.16661083698272705, -0...."
3,653432,False,False,-1,-1,-1,"[0.0987292230129242, 0.07315211743116379, 0.00..."
4,653436,False,False,-1,-1,-1,"[0.0987292230129242, 0.07315211743116379, 0.00..."


In [31]:
df = pd.merge(
    df_big,
    df_small,
    left_on="unique_id",
    right_on="review_id",
    how="left"
)
df.drop(columns=["review_id", "review_highlights"], inplace=True)

for col in ["positive_quality", "negative_quality"]:
    df[col] = df[col].fillna(False)
    
for col in ["noise_level", "group_size", "waiting_time"]:
    df[col] = df[col].fillna(-1).astype(int)
    
embedding_cols = [col for col in df.columns if "embedding" in col]

for col in embedding_cols:
    # Drop nulls safely, assuming each entry is a list/array or None/NaN
    non_null_embeddings = df[col].dropna()

    # Only proceed if there are any valid embeddings
    if len(non_null_embeddings) > 0:
        # Convert to stacked array and compute mean vector
        mean_embedding = np.stack(non_null_embeddings.values).mean(axis=0)

        # Replace nulls with mean vector
        def fill_embedding(x):
            if isinstance(x, (list, np.ndarray)):
                return x
            else:
                return mean_embedding

        df[col] = df[col].apply(fill_embedding)
        
df.head()

,timestamp,rating_value,local_guide,unique_id,review_text_embedding,positive_quality,negative_quality,noise_level,group_size,waiting_time,description_embedding
0,2020-12-04 10:39:16+00:00,4.0,True,0,"[-0.02132909931242466, 0.0855952724814415, 0.0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
1,2021-12-04 10:39:16+00:00,5.0,True,1,"[0.11749532073736191, 0.027359535917639732, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
2,2022-12-04 10:39:16+00:00,5.0,True,2,"[-0.0026724785566329956, 0.1681678593158722, 0...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
3,2021-12-04 10:39:16+00:00,5.0,True,3,"[0.11633721739053726, -0.0018988546216860414, ...",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."
4,2021-12-04 10:39:16+00:00,4.0,False,4,"[-0.1472267359495163, -0.14695429801940918, 0....",False,False,-1,-1,-1,"[0.0973015590350905, 0.048725716206544194, 0.0..."


In [32]:
df.to_parquet(os.path.join(feat_dir, "review.parquet"), index=False)

In [ ]:
""" Processing SERVICE """

In [53]:
df = pd.read_parquet(os.path.join(feat_dir, "tmp", "review_service.parquet"))
df.head()


,review_id,service,score
0,653221,Cibo,5/5
1,653221,Servizio,5/5
2,653221,Ambiente,5/5
3,653224,Cibo,5/5
4,653224,Servizio,5/5


In [35]:
prova = df["service"].unique()
prova

array(['Cibo', 'Servizio', 'Ambiente', 'Consigliato per i vegetariani',
       'Camere', 'Posizione'], dtype=object)

In [37]:
prova = df[df["service"] == "Consigliato per i vegetariani"]["score"].unique()
prova

array(['Vivamente consigliato', 'Abbastanza consigliato',
       'Non consigliato', 'Non so'], dtype=object)

In [54]:
veg_map = {
    "Vivamente consigliato": 5.0,
    "Abbastanza consigliato": 3.5,
    "Non consigliato": 1.5,
    "Non so": 2.5
}

def map_veg_score(row):
    if row['service'] == 'Consigliato per i vegetariani':
        return veg_map.get(row['score'], np.nan)  
    return row['score']  # leave as-is for other services

df['score'] = df.apply(map_veg_score, axis=1)


def parse_rating(value):
    try:
        parts = value.split('\n')  # Split if multiple ratings
        nums = [float(p.split('/')[0]) for p in parts]  
        return sum(nums) / len(nums)  
    except Exception:
        return value 
    
df["score"] = df["score"].apply(parse_rating)

df.head()

,review_id,service,score
0,653221,Cibo,5.0
1,653221,Servizio,5.0
2,653221,Ambiente,5.0
3,653224,Cibo,5.0
4,653224,Servizio,5.0


In [55]:
services = df["service"].drop_duplicates().tolist()
df_services = pd.DataFrame({
    "unique_id": range(len(services)),
    "name": services
})
df_services

,unique_id,name
0,0,Cibo
1,1,Servizio
2,2,Ambiente
3,3,Consigliato per i vegetariani
4,4,Camere
5,5,Posizione


In [56]:
df_tmp = df_services.copy()

mapping = dict(zip(df_services["name"], df_services["unique_id"]))
mapping

{'Cibo': 0,
 'Servizio': 1,
 'Ambiente': 2,
 'Consigliato per i vegetariani': 3,
 'Camere': 4,
 'Posizione': 5}

In [57]:
df_services_ok = compute_italian_embedding(df_services, "name")
df_services_ok.drop(columns=["name"], inplace=True)
df_services_ok.to_parquet(os.path.join(feat_dir, "service.parquet"), index=False)

df_services_ok.head()

Batches: 100%|██████████| 1/1 [00:00<00:00, 39.43it/s]


,unique_id,name_embedding
0,0,"[-0.26262152194976807, 0.1820594072341919, -0...."
1,1,"[-0.11518297344446182, -0.14848442375659943, -..."
2,2,"[0.3130197525024414, 0.2622040808200836, 0.189..."
3,3,"[0.05102507397532463, -0.03212948143482208, -0..."
4,4,"[0.5007088780403137, 0.10016030818223953, -0.0..."


In [59]:
df.head()

,review_id,service,score
0,653221,Cibo,5.0
1,653221,Servizio,5.0
2,653221,Ambiente,5.0
3,653224,Cibo,5.0
4,653224,Servizio,5.0


In [60]:
df["service_id"] = df["service"].map(mapping).astype(int)
df.drop(columns=["service"], inplace=True)
df.head()

,review_id,score,service_id
0,653221,5.0,0
1,653221,5.0,1
2,653221,5.0,2
3,653224,5.0,0
4,653224,5.0,1


In [14]:
df = df[['review_id', 'service_id', 'score']]
df.to_parquet(os.path.join(edgelist_dir, "review_is_about_service.parquet"), index=False)
df.head()

NameError: name 'df' is not defined

In [18]:
df_poi_service = pd.read_parquet(os.path.join(feat_dir, "tmp", "poi_service.parquet"))
df_poi_service.head()

,review_id,service,score
0,653229,Area parcheggio,Molti posti auto
1,653229,Opzioni per il parcheggio,Parcheggio senza costi
2,653253,Area parcheggio,Non so
3,653253,Opzioni per il parcheggio,Parcheggio senza costi
4,653274,Opzioni per il parcheggio,Il parcheggio non è disponibile


In [19]:
df_RP = pd.read_parquet(os.path.join(edges_dir, "review_is_about_poi.parquet"))
df_RP.head()

,unique_id_review,unique_id_poi
0,0,13976
1,1,13976
2,2,13976
3,3,13976
4,4,13976


In [20]:
df_merged = df_poi_service.merge(df_RP, left_on='review_id', right_on='unique_id_review', how='left')
df_merged = df_merged.rename(columns={'unique_id_poi': 'poi_id'})
df_merged.drop(columns=['unique_id_review', 'review_id'], inplace=True)
df_merged.head()

,service,score,poi_id
0,Area parcheggio,Molti posti auto,4736
1,Opzioni per il parcheggio,Parcheggio senza costi,4736
2,Area parcheggio,Non so,4862
3,Opzioni per il parcheggio,Parcheggio senza costi,4862
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742


In [9]:
df_poi_service["service"].unique()

#to_service_node_poi = ["Area parcheggio", "Opzioni per il parcheggio", "Attività consigliate", "Servizi", "Piatti vegetariani", "Regimi alimentari previsti", "Prenotazione", "Prenotazione consigliata", "Prenotazione"] #review_id, service_name, score    
#parcheggio_gratuito, parcheggio_a_pagamento, passeggiate, escursioni, tour_della_citta, centro_estetico, cucina_vegetariana, cucina_vegana, cucina_gluten_free, cucina_senza_lattosio, prenotazione
#free_parking, paid_parking, hikes, city_tour, beauty_center, vegetarian_cuisine, vegan_cuisine, gluten-free_cuisine, lactose-free_cuisine, reservation

array(['Area parcheggio', 'Opzioni per il parcheggio', 'Prenotazione',
       'Piatti vegetariani', 'Regimi alimentari previsti',
       'Prenotazione consigliata', 'Attività consigliate', 'Servizi'],
      dtype=object)

In [24]:
# PARCHEGGIO

def map_parking(row): #return free_parking, paid_parking
    service = row['service']
    value = row["score"]
    if service == "Area parcheggio" and value == "Molti posti auto":
        return True,False        
    elif service == "Opzioni per il parcheggio" and "senza costi" in value:
        return True, False
    elif service == "Opzioni per il parcheggio" and "a pagamento" in value:
        return False, True
    else:
        return False, False

df_merged[['free_parking', 'paid_parking']] = df_merged.apply(map_parking, axis=1, result_type='expand')

df_merged.head()

,service,score,poi_id,free_parking,paid_parking
0,Area parcheggio,Molti posti auto,4736,True,False
1,Opzioni per il parcheggio,Parcheggio senza costi,4736,True,False
2,Area parcheggio,Non so,4862,False,False
3,Opzioni per il parcheggio,Parcheggio senza costi,4862,True,False
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742,False,False


In [35]:
# ATTIVITA

def map_activities(row):
    service = row['service']
    value = row["score"]
    
    if service == "Attività consigliate":
        return True
    if service == "Servizi" and  "Tour" in value:
        return False
    return False

df_merged["excursion"] = df_merged.apply(map_activities, axis=1)

df_merged.head()

,service,score,poi_id,free_parking,paid_parking,reservation,excursion
0,Area parcheggio,Molti posti auto,4736,True,False,False,False
1,Opzioni per il parcheggio,Parcheggio senza costi,4736,True,False,False,False
2,Area parcheggio,Non so,4862,False,False,False,False
3,Opzioni per il parcheggio,Parcheggio senza costi,4862,True,False,False,False
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742,False,False,False,False


In [32]:
# PRENOTAZIONE
# city_tour, beauty_center, vegetarian_cuisine, vegan_cuisine, gluten-free_cuisine, lactose-free_cuisine,

def map_reservation(row): #reservation
    service = row['service']
    value = row["score"]

    if service == "Prenotazione consigliata" and value == "Sì":
        return True
    if service == "Prenotazione" and "Prenotazion" in value:
        return True
    
    return False

df_merged["reservation"] = df_merged.apply(map_reservation, axis=1)

df_merged.head()

,service,score,poi_id,free_parking,paid_parking,hikes,reservation
0,Area parcheggio,Molti posti auto,4736,True,False,False,False
1,Opzioni per il parcheggio,Parcheggio senza costi,4736,True,False,False,False
2,Area parcheggio,Non so,4862,False,False,False,False
3,Opzioni per il parcheggio,Parcheggio senza costi,4862,True,False,False,False
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742,False,False,False,False


In [36]:
# SPA

def map_spa(row): #
    service = row['service']
    value = row["score"]

    if service == "Servizi" and "Tour" not in value:
        return True
    
    return False

df_merged["spa"] = df_merged.apply(map_spa, axis=1)

df_merged.head()

,service,score,poi_id,free_parking,paid_parking,reservation,excursion,spa
0,Area parcheggio,Molti posti auto,4736,True,False,False,False,False
1,Opzioni per il parcheggio,Parcheggio senza costi,4736,True,False,False,False,False
2,Area parcheggio,Non so,4862,False,False,False,False,False
3,Opzioni per il parcheggio,Parcheggio senza costi,4862,True,False,False,False,False
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742,False,False,False,False,False


In [40]:
# vegetarian_cuisine, vegan_cuisine, gluten-free_cuisine, lactose-free_cuisine,

#CUCINA

def map_cucina(row): #
    
    vegetarian_cuisine = False
    vegan_cuisine = False
    gluten_free_cuisine = False 
    lactose_free_cuisine = False
    
    service = row['service']
    value = row["score"]

    if service == "Piatti vegetariani" and "vegetarian" in value:
        vegetarian_cuisine = True
    if service == "Regimi alimentari previsti" and "Vegetariano" in value:
        vegetarian_cuisine = True
    if service == "Regimi alimentari previsti" and "Vegano" in value:
        vegan_cuisine = True
    if service == "Regimi alimentari previsti" and "Senza glutine" in value:
        gluten_free_cuisine = True
    if service == "Regimi alimentari previsti" and "Senza lattosio" in value:
        lactose_free_cuisine = True
    
    return vegetarian_cuisine, vegan_cuisine, gluten_free_cuisine, lactose_free_cuisine

df_merged[["vegetarian_cuisine", "vegan_cuisine", "gluten-free_cuisine", "lactose-free_cuisine"]] = df_merged.apply(map_cucina, axis=1, result_type='expand')

df_merged.head()


,service,score,poi_id,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,Area parcheggio,Molti posti auto,4736,True,False,False,False,False,False,False,False,False
1,Opzioni per il parcheggio,Parcheggio senza costi,4736,True,False,False,False,False,False,False,False,False
2,Area parcheggio,Non so,4862,False,False,False,False,False,False,False,False,False
3,Opzioni per il parcheggio,Parcheggio senza costi,4862,True,False,False,False,False,False,False,False,False
4,Opzioni per il parcheggio,Il parcheggio non è disponibile,4742,False,False,False,False,False,False,False,False,False


In [42]:
df_merged.drop(["service", "score"], axis=1, inplace=True)
df_merged.head()

,poi_id,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,4736,True,False,False,False,False,False,False,False,False
1,4736,True,False,False,False,False,False,False,False,False
2,4862,False,False,False,False,False,False,False,False,False
3,4862,True,False,False,False,False,False,False,False,False
4,4742,False,False,False,False,False,False,False,False,False


In [44]:
df_merged_grouped = df_merged.groupby('poi_id', as_index=False).any()
df_merged_grouped.head()

,poi_id,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,1,True,False,False,False,False,False,False,False,False
1,12,True,False,False,False,False,False,False,False,False
2,13,True,False,False,False,False,False,False,False,False
3,22,True,False,False,False,False,False,False,False,False
4,27,True,False,False,False,False,False,False,False,False


In [50]:
df_merged_grouped.dtypes

poi_id                  int64
free_parking             bool
paid_parking             bool
reservation              bool
excursion                bool
spa                      bool
vegetarian_cuisine       bool
vegan_cuisine            bool
gluten-free_cuisine      bool
lactose-free_cuisine     bool
dtype: object

In [46]:
df_merged_grouped.to_parquet(os.path.join(feat_dir, "tmp", "poi_grouped.parquet"), index=False)

In [47]:
df_poi = pd.read_parquet(os.path.join(feat_dir, "poi.parquet"))
df_poi.head()


,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,avg_rating,votes_count,inexpensive,moderate,expensive,very_expensive,open24/7,title_embedding
0,1,1,40.508039,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,4.6,31,False,False,False,False,True,"[-0.0924561396241188, 0.3570820093154907, -0.1..."
1,2,427,40.522884,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,4.7,217,False,False,False,False,False,"[0.03634057939052582, 0.37386876344680786, 0.0..."
2,3,4,40.508718,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,4.5,2,False,False,False,False,True,"[0.2866106331348419, 0.33424651622772217, -0.0..."
3,4,4,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,4.6,41,True,False,False,False,False,"[0.6681708693504333, -0.4947139322757721, -0.1..."
4,5,93,40.510964,15.806366,False,4,None,True,False,False,3.8,61,False,False,False,False,True,"[0.4959424138069153, -0.09553607553243637, -0...."


In [52]:
df_final = pd.merge(
    df_poi, 
    df_merged_grouped, 
    how='left', 
    left_on='unique_id', 
    right_on='poi_id'
)

df_final = df_final.drop(columns='poi_id')
bool_cols = df_merged_grouped.columns.drop('poi_id')  # all boolean columns
df_final[bool_cols] = df_final[bool_cols].astype(bool)
df_final[bool_cols] = df_final[bool_cols].fillna(False)

df_final.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,...,title_embedding,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,1,1,40.508039,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,...,"[-0.0924561396241188, 0.3570820093154907, -0.1...",True,True,True,True,True,True,True,True,True
1,2,427,40.522884,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,...,"[0.03634057939052582, 0.37386876344680786, 0.0...",True,False,False,False,False,False,False,False,False
2,3,4,40.508718,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,...,"[0.2866106331348419, 0.33424651622772217, -0.0...",True,True,True,True,True,True,True,True,True
3,4,4,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,...,"[0.6681708693504333, -0.4947139322757721, -0.1...",True,True,True,True,True,True,True,True,True
4,5,93,40.510964,15.806366,False,4,None,True,False,False,...,"[0.4959424138069153, -0.09553607553243637, -0....",True,True,True,True,True,True,True,True,True


In [53]:
#main_image_embedding
embedding_cols = [col for col in df_final.columns if "embedding" in col]

for col in embedding_cols:
    # Drop nulls safely, assuming each entry is a list/array or None/NaN
    non_null_embeddings = df_final[col].dropna()

    # Only proceed if there are any valid embeddings
    if len(non_null_embeddings) > 0:
        # Convert to stacked array and compute mean vector
        mean_embedding = np.stack(non_null_embeddings.values).mean(axis=0)

        # Replace nulls with mean vector
        def fill_embedding(x):
            if isinstance(x, (list, np.ndarray)):
                return x
            else:
                return mean_embedding

        df_final[col] = df_final[col].apply(fill_embedding)
        
df_final.head()

,rank_absolute,total_photos,latitude,longitude,is_claimed,unique_id,main_image_embedding,restaurant,hotel,attraction,...,title_embedding,free_parking,paid_parking,reservation,excursion,spa,vegetarian_cuisine,vegan_cuisine,gluten-free_cuisine,lactose-free_cuisine
0,1,1,40.508039,15.812844,False,0,"[-0.19753423, 0.22135854, 0.1589038, -0.080560...",True,False,False,...,"[-0.0924561396241188, 0.3570820093154907, -0.1...",True,True,True,True,True,True,True,True,True
1,2,427,40.522884,15.785847,True,1,"[0.46170676, 0.29732698, 0.17882484, -0.351835...",True,False,False,...,"[0.03634057939052582, 0.37386876344680786, 0.0...",True,False,False,False,False,False,False,False,False
2,3,4,40.508718,15.811278,True,2,"[-0.031655762, 0.32317862, -0.25760216, 0.0752...",True,False,False,...,"[0.2866106331348419, 0.33424651622772217, -0.0...",True,True,True,True,True,True,True,True,True
3,4,4,40.507664,15.813661,True,3,"[-0.14220355, -0.040702403, -0.09907015, -0.24...",True,False,False,...,"[0.6681708693504333, -0.4947139322757721, -0.1...",True,True,True,True,True,True,True,True,True
4,5,93,40.510964,15.806366,False,4,"[-0.109207876, 0.08907696, -0.011386456, -0.04...",True,False,False,...,"[0.4959424138069153, -0.09553607553243637, -0....",True,True,True,True,True,True,True,True,True


In [54]:
df_final.to_parquet(os.path.join(feat_dir, "poi.parquet"), index=False)

ZONE

In [56]:
dfZ = pd.read_parquet(os.path.join(nodes_dir, "zone.parquet"))
dfZ.to_parquet(os.path.join(feat_dir, "zone.parquet"), index=False)
dfZ.head()

,unique_id,zone
0,0,9053426


GROUND TRUTH REVIEW

In [21]:
df_gt = pd.read_parquet(os.path.join(base_dir, "raw_data", f"review_{region_str}_filtered_by_user_output_llm.parquet"))
df_gt.shape

(53468, 16)

In [22]:
df_gt.head()

,cid,location_code,type,datetime,profile_url,review_text,timestamp,rating_value,reviews_count,photos_count,local_guide,review_highlights,unique_id,review_text_embedding,llm_output,llm_explanation
0,6526278041137503800,9053426,google_reviews,2023-12-04 10:39:53 +00:00,Z8SMxeefAI48R2x9XvuatlAo9/cz6/aRve/8T5mpScM=,None,2022-12-04 10:39:33 +00:00,4.0,25.0,23.0,True,None,20,None,None,None
1,13288306697958816570,9053426,google_reviews,2023-12-04 10:39:55 +00:00,ptBUM00JDZ3Fp48r96ReTjRfwW5jd6aey396I6pCrlU=,None,2023-10-04 10:39:55 +00:00,5.0,35.0,22.0,True,None,47,None,None,None
2,10300766960644890473,9053426,google_reviews,2023-12-04 10:39:58 +00:00,/fXVOQFxZ898Tkkwy+w6rVueVUOYQP1AnGlONo8efZU=,None,2019-12-04 10:39:39 +00:00,5.0,NaN,NaN,True,None,105,None,None,None
3,7713525035478443288,9053426,google_reviews,2023-12-04 10:40:03 +00:00,JgvCRT57Musz88y/jRm+B7OtFhuehEPwMuQOqgPVpRU=,Bello...peccato che duri poco,2017-12-04 10:37:44 +00:00,5.0,71.0,2.0,True,None,262,"[0.005026849452406168, 0.07046873867511749, 0....","{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",La recensione 'Bello...peccato che duri poco' ...
4,7713525035478443288,9053426,google_reviews,2023-12-04 10:40:03 +00:00,gh8fyZpBKDI5eTuOMeddhKdsxUfy9ICAO6kRl1FWdLE=,Da provare,2019-12-04 10:37:55 +00:00,4.0,89.0,87.0,True,None,278,"[0.2077694535255432, -0.07451275736093521, -0....","{'Animal Beach': None, 'Animal Lover': 0.0, 'A...","La recensione ""Da provare"" suggerisce un inter..."


In [23]:
df_ok = df_gt[["unique_id", "llm_output", "llm_explanation"]].copy()
df_ok["unique_id"] = df_ok["unique_id"].apply(lambda x: x - 1)
df_ok.head()

,unique_id,llm_output,llm_explanation
0,19,None,None
1,46,None,None
2,104,None,None
3,261,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",La recensione 'Bello...peccato che duri poco' ...
4,277,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A...","La recensione ""Da provare"" suggerisce un inter..."


In [24]:
df_ok = df_ok[~df_ok["llm_output"].isna()]
df_ok.to_parquet(os.path.join(base_dir, "original_data", "review_llm.parquet"), index=False)
df_ok.shape

(36672, 3)

In [26]:
df_ok["llm_output"].iloc[10]

{'Animal Beach': None,
 'Animal Lover': 0.0,
 'Animal Spiaggia': None,
 'Animale Spiaggia': 0.0,
 'Colto': 0.0,
 'Curioso': 0.6,
 'Disorganizzato': 0.0,
 'Esploratore': 0.0,
 'Famiglia': 0.2,
 'GenZ': 0.3,
 'Nomadi Digitali': 0.0,
 'Pigro': 0.0,
 'Romantico': 0.0,
 'Spirito Libero': 0.0,
 'Viaggiatore Affari': 0.0,
 'Viaggiatore Online': 0.1,
 'Viaggiatore Verde': 0.0,
 'spiegazione': None}

In [40]:
def safe_parse(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

df_new = df_ok[["unique_id", "llm_output"]].copy()
df_new["parsed_output"] = df_new["llm_output"].apply(safe_parse)
df_expanded = pd.json_normalize(df_new["parsed_output"])
df_expanded.drop(["Animal Beach", "Animal Spiaggia", "spiegazione"], axis=1, inplace=True)

translation_map = {
    'Curioso': 'curious',
    'Esploratore': 'explorer',
    'Colto': 'cultured',
    'Spirito Libero': 'free_spirit',
    'Romantico': 'romantic',
    'Famiglia': 'family_oriented',
    'Animale Spiaggia': 'beach_person',
    'Pigro': 'lazy',
    'Animal Lover': 'animal_lover',
    'Viaggiatore Online': 'online_traveler',
    'Disorganizzato': 'disorganized',
    'Viaggiatore Verde': 'eco_traveler',
    'GenZ': 'genZ',
    'Nomadi Digitali': 'digital_nomad',
    'Viaggiatore Affari': 'business_traveler'
}

df_expanded.rename(columns=translation_map, inplace=True)
df_final = pd.concat([df_new["unique_id"].reset_index(drop=True), df_expanded.reset_index(drop=True)], axis=1)
df_final["unique_id"].astype(int)
df_final.to_parquet(os.path.join(base_dir, "original_data", "review_llm.parquet"), index=False)

df_final.head()

,unique_id,animal_lover,beach_person,cultured,curious,disorganized,explorer,family_oriented,genZ,digital_nomad,lazy,romantic,free_spirit,business_traveler,online_traveler,eco_traveler
0,261,0.0,0.0,0.0,0.2,0.0,0.1,0.1,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.1
1,277,0.0,0.0,0.1,0.2,0.1,0.1,0.1,0.1,0.0,0.0,0.0,0.0,0.0,0.2,0.1
2,304,0.0,0.0,0.1,0.2,0.1,0.3,0.6,0.2,0.0,0.0,0.0,0.1,0.0,0.2,0.1
3,533,0.1,0.0,0.1,0.2,0.1,0.7,0.2,0.3,0.1,0.0,0.1,0.3,0.0,0.2,0.5
4,554,0.0,0.0,0.0,0.2,0.0,0.0,0.3,0.2,0.0,0.0,0.0,0.1,0.0,0.1,0.0


In [48]:
labels = sorted(list(df_final.columns[1:]))
labels_id = range(len(labels))

df_labels = pd.DataFrame({
    "label_name": labels,
    "label_id": labels_id
})

df_labels.to_parquet(os.path.join(base_dir, "original_data", "labels.parquet"), index=False)

df_labels


,label_name,label_id
0,animal_lover,0
1,beach_person,1
2,business_traveler,2
3,cultured,3
4,curious,4
5,digital_nomad,5
6,disorganized,6
7,eco_traveler,7
8,explorer,8
9,family_oriented,9


USER

In [50]:
dfUR = pd.read_parquet(os.path.join(edges_dir, "user_wrote_review.parquet"))
dfUR.head()

,unique_id_user,unique_id_review
0,182036,0
1,583934,1
2,356356,2
3,451564,3
4,460503,4


In [53]:
df_llm = pd.read_parquet(os.path.join(base_dir, "original_data", "review_llm.parquet"))
df_llm.head()

,unique_id,animal_lover,beach_person,cultured,curious,disorganized,explorer,family_oriented,genZ,digital_nomad,lazy,romantic,free_spirit,business_traveler,online_traveler,eco_traveler
0,261,0.0,0.0,0.0,0.2,0.0,0.1,0.1,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.1
1,277,0.0,0.0,0.1,0.2,0.1,0.1,0.1,0.1,0.0,0.0,0.0,0.0,0.0,0.2,0.1
2,304,0.0,0.0,0.1,0.2,0.1,0.3,0.6,0.2,0.0,0.0,0.0,0.1,0.0,0.2,0.1
3,533,0.1,0.0,0.1,0.2,0.1,0.7,0.2,0.3,0.1,0.0,0.1,0.3,0.0,0.2,0.5
4,554,0.0,0.0,0.0,0.2,0.0,0.0,0.3,0.2,0.0,0.0,0.0,0.1,0.0,0.1,0.0


In [55]:
df_merged = df_llm.merge(
    dfUR,
    how='left',
    left_on='unique_id',
    right_on='unique_id_review'
)

df_merged.drop(columns=['unique_id_review', 'unique_id'], inplace=True)
df_grouped = df_merged.groupby("unique_id_user", as_index=False).mean()

df_grouped.head()


,unique_id_user,animal_lover,beach_person,cultured,curious,disorganized,explorer,family_oriented,genZ,digital_nomad,lazy,romantic,free_spirit,business_traveler,online_traveler,eco_traveler
0,56,0.000000,0.0,0.021918,0.208219,0.109589,0.008219,0.184932,0.230137,0.004110,0.013699,0.020548,0.108219,0.00137,0.154795,0.039726
1,464,0.004000,0.0,0.026000,0.128000,0.062000,0.016000,0.176000,0.142000,0.000000,0.010000,0.012000,0.048000,0.01200,0.118000,0.020000
2,624,0.004545,0.0,0.040909,0.213636,0.068182,0.063636,0.286364,0.227273,0.009091,0.022727,0.134091,0.140909,0.00000,0.190909,0.077273
3,735,0.000000,0.0,0.000000,0.000000,0.100000,0.000000,0.200000,0.000000,0.000000,0.600000,0.300000,0.000000,0.00000,0.100000,0.000000
4,1016,0.000000,0.0,0.620000,0.620000,0.140000,0.140000,0.340000,0.400000,0.080000,0.120000,0.160000,0.060000,0.00000,0.500000,0.220000


In [56]:
df_user_labels = df_grouped.copy()
df_user_labels["label"] = df_user_labels.drop(columns=["unique_id_user"]).idxmax(axis=1)

df_result = df_user_labels[["unique_id_user", "label"]].copy()

dict_labels = df_labels.set_index(df_labels.columns[0])[df_labels.columns[1]].to_dict()

df_result["label_id"] = df_result["label"].map(dict_labels)

df_result.to_parquet(os.path.join(base_dir, "original_data", "user_labels.parquet"), index=False)

df_result.head()

,unique_id_user,label,label_id
0,56,genZ,11
1,464,family_oriented,9
2,624,family_oriented,9
3,735,lazy,12
4,1016,cultured,3


In [57]:
df_result.to_parquet(os.path.join(base_dir, "original_data", "user_labels.parquet"), index=False)

In [51]:
dfU = pd.read_parquet(os.path.join(nodes_dir, "user.parquet"))
dfU.head()

,profile_url,llm_profile_ground_truth,unique_id
0,hZkGCMpnXFZ+ES6l8UJOI+G4yTYl77t/4L2R7bHkfxE=,None,0
1,u0+BAfG1ytT8PKIG6xf7GzNOApWcmn0JQOkLoczsOkg=,None,1
2,lbaCbeDx8H7PnANKwp4PYtCh8IAGHfxOSxH2RuTz+CI=,None,2
3,oJN2MlXD549CeljkWSeKH17SUv1Mu+Kzcm0koTrEOS0=,None,3
4,PWyMDlMvEN91/2ntyV1VORfZyOEIuElr57hH/xHrHLQ=,None,4


In [ ]:
dfU = encoding_short_text(dfU, "profile_url", target_dim=256)
dfU.drop(["llm_profile_ground_truth"], axis=1, inplace=True)
dfU.to_parquet(os.path.join(nodes_dir, "user.parquet"), index=False)

In [4]:
dfU = pd.read_parquet(os.path.join(nodes_dir, "user.parquet"))

dfU.head()

,unique_id,profile_url_encoded
0,0,"[0.32227203249931335, 0.16211727261543274, 0.1..."
1,1,"[0.13920839130878448, 0.1286950558423996, -0.0..."
2,2,"[0.235089510679245, -0.03374931961297989, 0.09..."
3,3,"[0.10278420895338058, -0.005469620227813721, -..."
4,4,"[-0.0021424219012260437, -0.002127068117260933..."
